**Part-4 : README**

** Task 1 — Database Schema Design**

-I created a MySQL database named `trendpulse` with three related tables:
`categories`, `videos`, and `trending_records`.  
-Primary keys uniquely identify records, while foreign keys connect videos to categories and trending records to videos.  
-This relational design reduces repeated information and supports SQL joins.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print(df.shape)
print(df.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(17000, 9)
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [ ]:
import glob

files = glob.glob("/content/drive/MyDrive/**/*.csv", recursive=True)
print(files[:20])

['/content/drive/MyDrive/Masai Assignment/assignmentsUntitled folder/Book(Sheet1).csv', '/content/drive/MyDrive/Masai Assignment/Content/dataset/GBvideos.csv', '/content/drive/MyDrive/Masai Assignment/Content/dataset/USvideos.csv', '/content/drive/MyDrive/Masai Assignment/Content/dataset/INvideos.csv', '/content/drive/MyDrive/Masai Assignment/Content/dataset/categories.csv']


## Task 1 — SQL Schema Design

I created a MySQL database named `trendpulse` using DB Fiddle.

The database contains three related tables:
- categories
- videos
- trending_records

Primary keys uniquely identify records, and foreign keys connect the tables.

### SQL Schema

Paste the complete CREATE DATABASE and CREATE TABLE SQL code here inside a ```sql code block.

### Result

The SHOW TABLES query successfully displayed all three tables.

**Schema (MySQL v9)**

    CREATE DATABASE IF NOT EXISTS trendpulse;
    USE trendpulse;
    CREATE TABLE categories (category_id INT PRIMARY KEY, category_name VARCHAR(255));
    
    CREATE TABLE videos (
    video_id VARCHAR(100) PRIMARY KEY,
    title TEXT, channel_title VARCHAR(255),
    category_id INT, publish_time DATETIME,
    FOREIGN KEY (category_id) REFERENCES categories(category_id));
    
    CREATE TABLE trending_records (
    record_id INT AUTO_INCREMENT PRIMARY KEY,
    video_id VARCHAR(100), country VARCHAR(10),
    trending_date DATE, views BIGINT, likes BIGINT,
    dislikes BIGINT, comment_count BIGINT,
    FOREIGN KEY (video_id) REFERENCES videos(video_id));USE trendpulse;
    SELECT COUNT(*) FROM categories;
    SELECT COUNT(*) FROM videos;
    SELECT COUNT(*) FROM trending_records;
    
    SELECT video_id, views AS total_views
    FROM trending_records
    WHERE country = 'IN'
    ORDER BY views DESC
    LIMIT 10;
    
    SELECT DISTINCT country
    FROM trending_records;
    
    SELECT DISTINCT category_id
    FROM videos
    ORDER BY category_id;

---

**Query #1**

    USE trendpulse;

There are no results to be displayed.

---
**Query #2**

    SHOW TABLES;

| Tables_in_trendpulse |
| -------------------- |
| categories           |
| trending_records     |
| videos               |

---
**Query #3**

    SELECT COUNT(*) FROM categories;

| COUNT(*) |
| -------- |
| 0        |

---
**Query #4**

    SELECT COUNT(*) FROM videos;

| COUNT(*) |
| -------- |
| 0        |

---
**Query #5**

    SELECT COUNT(*) FROM trending_records;

| COUNT(*) |
| -------- |
| 0        |

---

[View on DB Fiddle](https://www.db-fiddle.com/)

**Task-2: Data Loading**

I prepared the trending records table by selecting video IDs, trending dates, views, likes, dislikes, and comment counts. I added the country code GB and a unique record ID for each row.
I extracted unique category IDs from the trending video dataset. Category names will be mapped using the original YouTube category metadata file.

prepared three relational datasets: categories, videos, and trending records. I exported them as CSV files using Pandas with index=False for database loading.



In [ ]:
import pandas as pd

categories.to_sql("categories", conn, if_exists="replace", index=False)
videos.to_sql("videos", conn, if_exists="replace", index=False)
records.to_sql("trending_records", conn, if_exists="replace", index=False)

print("All 3 tables loaded!")

All 3 tables loaded!


In [ ]:
for table in ["categories", "videos", "trending_records"]:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table}", conn)
    print(table, count.iloc[0, 0])

categories 14
videos 4113
trending_records 15075


In [ ]:
for table in ["categories", "videos", "trending_records"]:
    count = cursor.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]
    print(table, count)

categories 14
videos 4113
trending_records 15075


In [ ]:
query = """
SELECT video_id, views AS total_views
FROM trending_records
WHERE country = 'GB'
ORDER BY views DESC
LIMIT 10;
"""
print(pd.read_sql(query, conn))

      video_id  total_views
0  RdqSDDng84n    360760164
1  QYwpwP6y0Eb    343297002
2  RdqSDDng84n    286714357
3  QYwpwP6y0Eb    282812154
4  WQEWsMmimzZ    255362869
5  RdqSDDng84n    236275528
6  QYwpwP6y0Eb    230704137
7  RdqSDDng84n    210747294
8  WQEWsMmimzZ    200444300
9  QYwpwP6y0Eb    181785943


**Task 2 — Data Loading Results**

I loaded the GB trending video dataset into SQLite using Python and Pandas.

Verified Row Counts:

Categories: 14

Videos: 4,113

Trending Records: 15,075

I verified the counts using SQL SELECT COUNT(*) queries.

In [ ]:
query = """
SELECT DISTINCT country
FROM trending_records;
"""
print(pd.read_sql(query, conn))

  country
0      GB


**Task 3 — Basic SQL Queries**

I used SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, and column aliases to identify the top 10 trending records by views, unique countries, and unique video categories.

**Task 4 — Grouping and Aggregation**

I used GROUP BY and AVG() to identify categories with average views above 100,000. I also used COUNT(DISTINCT) and HAVING to identify channels with at least five unique trending videos.

The queries were executed using SQLite on the GB trending videos dataset. Actual query outputs are included below each query.

In [ ]:
query = """
SELECT video_id, views AS total_views
FROM trending_records
WHERE country = 'GB'
ORDER BY views DESC
LIMIT 10;
"""
print(pd.read_sql(query, conn))

In [ ]:
query = """
SELECT DISTINCT country
FROM trending_records;
"""
print(pd.read_sql(query, conn))

In [ ]:
query = """
SELECT DISTINCT category_id
FROM videos
ORDER BY category_id;
"""
print(pd.read_sql(query, conn))

In [ ]:
#Task 4 — GROUP BY and HAVING:

query = """
SELECT v.category_id, ROUND(AVG(t.views), 2) AS avg_views
FROM videos v
JOIN trending_records t ON v.video_id = t.video_id
GROUP BY v.category_id
HAVING AVG(t.views) > 100000;
"""
print(pd.read_sql(query, conn))

In [ ]:
query = """
SELECT v.channel_title,
COUNT(DISTINCT v.video_id) AS total_videos
FROM videos v
JOIN trending_records t ON v.video_id = t.video_id
GROUP BY v.channel_title
HAVING COUNT(DISTINCT v.video_id) >= 5;
"""
print(pd.read_sql(query, conn))

**Task 5 — Excel Workbook Export:**

I exported the cleaned data into trendpulse_final.xlsx using Pandas.

The workbook includes:

Records — Trending video statistics.

Videos — Unique video details.

Lookup — Category ID and category name mapping.

Pivot Summary — Average views by category and country.

Summary — Spreadsheet calculations.

Pivot Analysis

Rows: Category Name

Columns: Country

Values: Views

Aggregation: Average

The Pivot Summary was generated using Pandas. It is a calculated summary rather than a native Excel PivotTable.



In [ ]:
import pandas as pd

with pd.ExcelWriter("trendpulse_final.xlsx") as writer:
    records.to_excel(writer, sheet_name="Records", index=False)
    videos.to_excel(writer, sheet_name="Videos", index=False)
    categories.to_excel(writer, sheet_name="Lookup", index=False)
    pivot.to_excel(writer, sheet_name="Pivot Summary")

**Task 6 — VLOOKUP and Spreadsheet Formulas**

VLOOKUP

=VLOOKUP(I2,Lookup!$A$2:$B$15,2,FALSE)

Retrieves the category name using its category ID.

COUNTIF

=COUNTIF(Records!H2:H15076,"GB")

Counts GB trending records.

SUM

=SUM(Records!D2:D15076)

Calculates total views.

AVERAGE

=AVERAGE(Records!D2:D15076)

Calculates average views

In [ ]:
from openpyxl import load_workbook

wb = load_workbook("trendpulse_final.xlsx")
ws = wb["Records"]
ws["K1"] = "Category Lookup"
for i in range(2, ws.max_row + 1):
    ws[f"K{i}"] = f"=VLOOKUP(I{i},Lookup!$A$2:$B$15,2,FALSE)"
wb.save("trendpulse_final.xlsx")

**Task 7 — Filters, Sorting and Conditional Formatting**

Filters were added to the Records sheet.

A descending sort condition was configured for the Views column.

Conditional formatting was applied to highlight the top 10% of views in yellow.

Range: Records!D2:D15076

Rule: Top 10% of values.



In [ ]:
from openpyxl.formatting.rule import Top10Rule
from openpyxl.styles import PatternFill

wb = load_workbook("trendpulse_final.xlsx")
ws = wb["Records"]
ws.auto_filter.ref = ws.dimensions
ws.auto_filter.add_sort_condition(f"D2:D{ws.max_row}", descending=True)
yellow = PatternFill("solid", fgColor="FFFF00")
ws.conditional_formatting.add(f"D2:D{ws.max_row}", Top10Rule(rank=10, percent=True, fill=yellow))
wb.save("trendpulse_final.xlsx")

**Task 8 — Named Range**

I created a named range called ViewsData.

Range: Records!$D$2:$D$15076

Formula:

=AVERAGE(ViewsData)

This calculates the average number of views using the named range.

In [ ]:
from openpyxl.workbook.defined_name import DefinedName

wb = load_workbook("trendpulse_final.xlsx")
ws = wb["Records"]
name = DefinedName("ViewsData", attr_text=f"'Records'!$D$2:$D${ws.max_row}")
wb.defined_names.add(name)
summary = wb["Summary"]
summary["A5"] = "Named Range Average"
summary["B5"] = "=AVERAGE(ViewsData)"
wb.save("trendpulse_final.xlsx")

**Task 9 — SQL and Excel Cross-check**

I compared two metrics between the SQLite database and the Excel workbook.

Metric 1: Total trending records

Metric 2: Total views

SQL query:

SELECT COUNT(*) AS total_records,
       SUM(views) AS total_views
FROM trending_records;

Excel calculations:

=COUNT(Records!D2:D15076)

=SUM(Records!D2:D15076)



In [ ]:
import sqlite3

con = sqlite3.connect("/content/drive/MyDrive/trendpulse.db")
sql = pd.read_sql("SELECT COUNT(*) AS count, SUM(views) AS views FROM trending_records", con)
excel = pd.read_excel("trendpulse_final.xlsx", sheet_name="Records")
print("SQL Count:", sql["count"][0])
print("Excel Count:", len(excel))
print("SQL Views:", sql["views"][0])
print("Excel Views:", excel["views"].sum())